In [ ]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ to sys.path so `import bowaka_v2_lab` works regardless of
# how the notebook is launched (jupyter / papermill / pytest).
import os
import sys
from pathlib import Path

_here = Path.cwd()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        sys.path.insert(0, str(_candidate / "src"))
        break
import bowaka_v2_lab  # noqa: F401
print("bowaka_v2_lab", bowaka_v2_lab.__version__)


In [ ]:
# Papermill parameter cell.
CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_backtest_smoke.yml'


# 05 — Single Config Backtest

In [ ]:
import datetime as _dt
import pandas as pd
from pathlib import Path
from bowaka_v2_lab.config import load_config, BowakaV2Paths
from bowaka_v2_lab.config.models import BowakaV2Config
from bowaka_v2_lab.sim.backtester import run_backtest
from bowaka_v2_lab.sim.replay_fixtures import synthetic_universe, synthetic_daily_cache
cfg = load_config(CONFIG_PATH)
validated = BowakaV2Config.model_validate(cfg)
paths = BowakaV2Paths.from_config(validated, repo_root=Path('.').resolve())
sessions = [_dt.date(2024, 9, 4)]
syms = cfg.get('universe', {}).get('symbols') or ['AAA','BBB','CCC']
universe = {sessions[0]: synthetic_universe(syms)}
daily_cache = {sessions[0]: synthetic_daily_cache(syms)}
def minute_supplier(sym, ts):
    rows = []
    for i in range(30):
        rows.append({'timestamp': pd.Timestamp('2024-09-04 13:30:00', tz='UTC') + pd.Timedelta(minutes=i),
                      'open': 100+i*0.1, 'high': 100+i*0.2, 'low': 99.5, 'close': 100+i*0.15, 'volume': 5000.0})
    return pd.DataFrame(rows)
def daily_supplier(sym, d):
    return pd.DataFrame([{'symbol': sym, 'session_date': d, 'open': 100.0, 'high': 110.0, 'low': 98.0, 'close': 108.0, 'volume': 100000}])
result = run_backtest(cfg=cfg, sessions=sessions,
  scan_times_per_session=lambda d: [pd.Timestamp(f'{d}T14:00:00', tz='UTC')],
  universe_snapshot_by_session=universe, daily_cache_by_session=daily_cache,
  minute_bars_supplier=minute_supplier, daily_bars_supplier=daily_supplier,
  initial_bankroll=10_000.0, paths=paths)
print('run_id:', result.run_id)
print('run_dir:', result.run_dir)
print('summary:', result.summary)
